# 02 — Protenix Baseline Inference

Run Protenix (AF3) pre-trained model on validation set to establish baseline TM-score.

In [ ]:
# === Colab Setup Cell ===
!pip install kaggle -q
import os
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

REPO_URL = "https://github.com/YOUR_USER/3drna_cc.git"  # <-- UPDATE THIS
if not os.path.exists('/content/3drna_cc'):
    !git clone {REPO_URL} /content/3drna_cc
%cd /content/3drna_cc

from src.setup import setup_environment
setup_environment()

## Step 1: Prepare Protenix Input JSONs

In [ ]:
from src.data.loader import load_sequences, load_labels
from src.data.featurizer import build_all_inputs
from src.config import OUTPUT_DIR

val_seq = load_sequences(split='val')
val_labels = load_labels(split='val')
print(f"Validation targets: {len(val_seq)}")

# Build Protenix input JSONs for all validation targets
input_dir = OUTPUT_DIR / 'val_inputs'
input_paths = build_all_inputs(val_seq, output_dir=input_dir, n_seeds=5)
print(f"Created {len(input_paths)} input JSONs in {input_dir}")

## Step 2: Run Protenix Inference

In [ ]:
from src.model.protenix_runner import ProtenixRunner
import numpy as np

runner = ProtenixRunner(device='cuda')

# Run inference on all validation targets
results = runner.predict_all(input_dir, n_seeds=5)
print(f"\nPredicted {len(results)} targets")
for tid, coords_list in results.items():
    print(f"  {tid}: {len(coords_list)} structures, "
          f"{coords_list[0].shape if coords_list else 'N/A'} residues")

## Step 3: Compute Baseline TM-scores

In [ ]:
from src.ensemble.tm_score import compute_tm_score, best_of_n_tm_score
from src.data.loader import labels_to_coords

tm_scores = []

for tid, preds in results.items():
    if not preds:
        print(f"  {tid}: NO PREDICTIONS")
        continue
    
    ref_coords = labels_to_coords(val_labels, tid)
    if len(ref_coords) == 0:
        print(f"  {tid}: no reference coords")
        continue
    
    # best-of-5 TM-score
    best_tm, best_idx = best_of_n_tm_score(preds[:5], ref_coords)
    tm_scores.append({'target_id': tid, 'best_tm': best_tm, 'best_model': best_idx})
    print(f"  {tid}: best-of-5 TM = {best_tm:.4f} (model {best_idx})")

import pandas as pd
tm_df = pd.DataFrame(tm_scores)
print(f"\n{'='*60}")
print(f"BASELINE RESULTS")
print(f"Mean best-of-5 TM-score: {tm_df['best_tm'].mean():.4f}")
print(f"Median: {tm_df['best_tm'].median():.4f}")
print(f"Min: {tm_df['best_tm'].min():.4f}, Max: {tm_df['best_tm'].max():.4f}")

## Step 4: Geometry Validation

In [ ]:
from src.postprocess.geometry_check import validate_geometry, print_geometry_report

# Check geometry of a few predictions
for tid, preds in list(results.items())[:3]:
    if not preds:
        continue
    print(f"\n--- {tid} (model 0) ---")
    report = validate_geometry(preds[0])
    print_geometry_report(report)